# Silver Layer — CRM Product Info
Clean and normalize `crm_prd_info`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]
VOLUME = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Read Bronze Table

In [ ]:
df = session.table(f"{SCHEMA}.crm_prd_info")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType, DateType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Product Key Parsing

In [ ]:
df = df.with_column("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), F.lit("-"), F.lit("_")))
df = df.with_column("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))

### Cost Cleanup

In [ ]:
df = df.with_column("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))

### Product Line Normalization

In [ ]:
df = df.with_column(
    "prd_line",
    F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
     .when(F.upper(F.col("prd_line")) == "R", "Road")
     .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
     .when(F.upper(F.col("prd_line")) == "T", "Touring")
     .otherwise("n/a")
)

### Date Casting

In [ ]:
df = df.with_column("prd_start_dt", F.col("prd_start_dt").cast(DateType()))

### Rename Columns

In [ ]:
RENAME_MAP = {
    "prd_id":       "product_id",
    "cat_id":       "category_id",
    "prd_key":      "product_number",
    "prd_nm":       "product_name",
    "prd_cost":     "product_cost",
    "prd_line":     "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt":   "end_date",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"{SCHEMA}.crm_products", mode="overwrite")
print("crm_products OK")

## Verify

In [ ]:
session.table(f"{SCHEMA}.crm_products").limit(5).show()